<a href="https://colab.research.google.com/github/tougheye/Data_processing/blob/main/Doctor_Name_find_in_99.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Questions - How to resolve if someone has multi campus appointments but is supervisor in one but not the other


The goal is to compare the campaign list of doctors against the most recent 99 list and identify anyone who is:

*   No longer at UC (not on the 99 list at all)
*   On the 99 list but not in a job title of interest (See "Job Titles of Interest" tab on the link above)
*   On the 99 list but listed as a manager or supervisor
*   On the 99 list but at 0% FTE

Import Doctor list

In [76]:
import pandas as pd

doctor_list_file = pd.ExcelFile("/content/drive/MyDrive/External Organizing/Doctors/Doctor_99_Match/Doctor Statewide List 6_4_26.xlsx")
file_sheet_names = doctor_list_file.sheet_names
print(file_sheet_names)


['Doctor Statewide Legislative Li', 'Job Titles of Interest']


In [77]:
doctor_list_df = doctor_list_file.parse(file_sheet_names[0])
doctor_list_df.shape

(8824, 34)

In [128]:
doctor_list_df[doctor_list_df['Sheet ID'] == 12732]

,Campus,Job Title,Department,First/Preferred Name,Middle Name,Last Name,Pronouns,UC Provided Work Location,Building,Room,...,Research Notes,Hire Date,Job Expected End Date,Employee Class Description,FTE,Job Hourly Rate,Employee Relation Description,Location/Business Unit,UC Employee Name,Sheet ID
3289,ucsd,HS ASSOC CLIN PROF-HCOMP,NEUROSCIENCES,David,JIN,Lee,NaN,"9444 MEDICAL CENTER DR, East Campus Office Bui...",ACTRI Bldg,NaN,...,NaN,2012-07-01 00:00:00,2026-06-30 00:00:00,Academic: Faculty,1,86.21,"Manager, Not Confidential",SDCMP,"Lee,David",12732


In [78]:
titles_of_interest_df = doctor_list_file.parse(file_sheet_names[1])


In [79]:
titles_of_interest = titles_of_interest_df['Job Title'].str.upper().str.strip().to_list()
titles_of_interest

['ASC PHYSCN',
 'ASC PHYSCN DIPLOMATE',
 'ASST ADJ PROF-HCOMP',
 'AST PHYSCN',
 'EXAMING PHYSCN',
 'HS ASSOC CLIN PROF-AY',
 'HS ASSOC CLIN PROF-FY',
 'HS ASSOC CLIN PROF-HCOMP',
 'HS ASSOC CLIN PROF-SFT-VM',
 'HS ASST CLIN PROF-AY',
 'HS ASST CLIN PROF-FY',
 'HS ASST CLIN PROF-HCOMP',
 'HS ASST CLIN PROF-HCOMPC',
 'HS ASST CLIN PROF-SFT-VM',
 'HS CLIN INSTR-FY',
 'HS CLIN INSTR-HCOMP',
 'HS CLIN PROF-AY',
 'HS CLIN PROF-FY',
 'HS CLIN PROF-HCOMP',
 'HS CLIN PROF-SFT-VM',
 'NATUROPATHIC DOCTOR EX',
 'PHYSCN ADM 3',
 'PHYSCN DIPLOMATE SR',
 'PHYSCN SHS 5 LIMITED HOURS',
 'PHYSCN SHS MGR 3',
 'PHYSCN SR',
 'STF PHYSCN']

Import Recent 99 list

In [80]:
latest_99_file = pd.ExcelFile("/content/drive/MyDrive/Data/Unit_99_Title_Claim/Historical 99 Data/UPTE_99_as_of_6_3_2026.xlsx")
latest_99_file_sheet_names = latest_99_file.sheet_names
print(latest_99_file_sheet_names)

['Sheet1']


In [81]:
latest_99_df = latest_99_file.parse(latest_99_file_sheet_names[0])
latest_99_df.shape

(172908, 25)

First filter the 99 list for only the titles of interest

In [82]:
latest_99_title_interest_df = latest_99_df[latest_99_df['Job Code Description'].str.upper().str.strip().isin(titles_of_interest)]
latest_99_title_interest_df.shape

(13100, 25)

Eliminate FTE = 0s

In [83]:
latest_99_of_interest_df = latest_99_title_interest_df[latest_99_title_interest_df['FTE'] > 0]
latest_99_of_interest_df.shape

(8962, 25)

Eliminate the managers and supervisors from the title of interest DF

In [84]:
EER_Desc_list = latest_99_of_interest_df['Employee Relation Description'].unique()

EER_mgr_supes = [x for x in EER_Desc_list if str(x).startswith('Manager') or str(x).startswith('Supervisor')]
EER_mgr_supes
# print(len(EER_mgr_supes), len(EER_Desc_list))


['Manager, Not Confidential',
 'Supervisor, Not Confidential',
 'Supervisor, Confidential',
 'Manager, Confidential']

In [85]:
# Create the organizing target dataframe getting rid of the managers and supervisors from title of interest df
latest_99_organizing_target_df = latest_99_of_interest_df[~latest_99_of_interest_df['Employee Relation Description'].isin(EER_mgr_supes)]
latest_99_organizing_target_df.shape

(7708, 25)

In [86]:
latest_99_organizing_target_df = latest_99_organizing_target_df.reset_index()

Create function to standardize email addresses on both df

In [88]:
def standardize_emails(df, email_col):
    df[email_col] = (
        df[email_col]
        .astype(str)
        .str.lower()
        .str.strip()
        .str.replace(r'\s+', '', regex=True) # Removes internal spaces
        .str.replace(r'\.(?=.*@)', '', regex=True) # Removes dots before the '@' symbol
    )
    return df

Merge based on UC emails. Will need multiple rounds since there are UC email 1, 2, 3 fields

In [89]:
# put the relevant columns in a list
rel_99_cols = ['index','Location/Business Unit', 'Employee Lived Name',
       'Job Code', 'Job Code Description', 'EE Work Email Address', 'Department Description', 'Employee Class Description',
       'Most Recent Date of Hire in Empl Rcd', 'FTE', 'Employee Relation Description']

In [90]:
latest_99_organizing_target_df = standardize_emails(latest_99_organizing_target_df, 'EE Work Email Address')
doctor_list_df = standardize_emails(doctor_list_df, 'UC Email')
doctor_list_df = standardize_emails(doctor_list_df, 'UC Email 2')
doctor_list_df = standardize_emails(doctor_list_df, 'UC Email 3')

Merge the lists based on email addresses

In [91]:
doctor_list_merge_email1_df = doctor_list_df.merge(latest_99_organizing_target_df[rel_99_cols], left_on=['UC Email'], right_on=['EE Work Email Address'], suffixes=('_doc', '_99'))
doctor_list_merge_email1_df.shape

(6408, 45)

In [92]:
doctor_list_merge_email1_df['matched_on'] = 'UC Email'

In [93]:
doctor_list_merge_email1_df['Sheet ID'].nunique()

6319

In [94]:
doctor_list_df['Sheet ID'].nunique()

8817

8817 unique Sheet IDs in the original doctor list.


*   7449 of those were matched based on UCemail1 3067 of them matched on multiple UC emails
*   1325 matched based on the UC Email 2. 216 of them matched on multiple UC emails

None matched on UC email 3.

In [95]:
# separate the sheet IDs in the original doctor list that did not match on the email1
doct_non_match_email1_df = doctor_list_df[~doctor_list_df['Sheet ID'].isin(doctor_list_merge_email1_df['Sheet ID'])]
doct_non_match_email1_df.shape

(2504, 34)

In [96]:
doctor_list_merge_email2_df = doct_non_match_email1_df.merge(latest_99_organizing_target_df[rel_99_cols], left_on=['UC Email 2'], right_on=['EE Work Email Address'], suffixes=('_doc', '_99'))
doctor_list_merge_email2_df.shape

(1022, 45)

In [97]:
doctor_list_merge_email2_df['matched_on'] = 'UC Email 2'

In [98]:
doctor_list_merge_email2_df['Sheet ID'].nunique()

1020

Create the master doctor list that are matched on email addresses

In [99]:
doct_email_matches_df = pd.concat([doctor_list_merge_email1_df, doctor_list_merge_email2_df])
doct_email_matches_df.shape

(7430, 46)

In [100]:
doct_email_matches_df['Sheet ID'].nunique()

7339

From the email matched master list, drop the sheetIDs that don't have valid. 1076 out of the 11335 initial matches will be dropped. That leaves 8,488 unique Sheet IDs that matched both on email addresses and names

In [101]:
False_positives = doct_email_matches_df[doct_email_matches_df['EE Work Email Address'] == "notprovided"]
False_positives.shape

(11, 46)

In [102]:
doct_email_matches_df = doct_email_matches_df.drop(False_positives.index)
doct_email_matches_df.shape

(7419, 46)

In [103]:
doct_email_matches_df['Sheet ID'].nunique()

7338

AFTER Dropping unmatched names from the email matches, 8432 unique sheet IDs are left

In [ ]:
doct_email_matches_df.to_excel('doctor_email_matches_w_ind_no_0_FTE.xlsx', index=False)

KeyboardInterrupt: 

In [104]:
doct_email_non_matches_df = doctor_list_df[~doctor_list_df['Sheet ID'].isin(doct_email_matches_df['Sheet ID'])]
doct_email_non_matches_df.shape

(1485, 34)

Merge Match the unmatched ones based on names

202 unique sheet IDs matched on 301 names

In [105]:
def standardize_names(df, name_col):
    df[f'{name_col}_std'] = (
        df[name_col]
        .astype(str)
        .str.lower()
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True) # Removes internal spaces
    )
    return df

In [106]:
doct_email_non_matches_df = standardize_names(doct_email_non_matches_df, 'UC Employee Name')
doct_email_non_matches_df.columns

/tmp/ipykernel_1738/938881767.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[f'{name_col}_std'] = (


Index(['Campus', 'Job Title', 'Department', 'First/Preferred Name',
       'Middle Name', 'Last Name', 'Pronouns', 'UC Provided Work Location',
       'Building', 'Room', 'Location notes: remote, office hours, etc.',
       'Personal Email', 'Personal Email 2', 'Personal Email 3', 'UC Email',
       'UC Email 2', 'UC Email 3', 'Cell Phone', 'Home Phone', 'Work Phone',
       'LinkedIn URL', 'Other Website(s)', 'Home Address', 'Age',
       'Research Notes', 'Hire Date', 'Job Expected End Date',
       'Employee Class Description', 'FTE', 'Job Hourly Rate',
       'Employee Relation Description', 'Location/Business Unit',
       'UC Employee Name', 'Sheet ID', 'UC Employee Name_std'],
      dtype='object')

In [107]:
latest_99_organizing_target_df = standardize_names(latest_99_organizing_target_df, 'Employee Lived Name')
latest_99_organizing_target_df.columns

Index(['index', 'Location/Business Unit', 'Employee Lived Name', 'Job Code',
       'Job Code Description', 'Most Recent Date of Hire in Empl Rcd',
       'Job Expected End Date', 'Employee Class', 'Employee Class Description',
       'FTE', 'Job Hourly Rate', 'Annual Salary', 'Department ID',
       'Department Description', 'Location Code', 'Location Code Description',
       'EE Work Location Address 1', 'EE Work Location Address 2',
       'EE Work Location Address 3', 'EE Work Location Floor',
       'EE Work Location City', 'EE Work Location State',
       'EE Work Location Zip Code', 'EE Work Email Address', 'EE Work Phone',
       'Employee Relation Description', 'Employee Lived Name_std'],
      dtype='object')

In [108]:
rel_99_cols.append('Employee Lived Name_std')
# rel_99_cols
email_unmatch_name_match_df = doct_email_non_matches_df.merge(latest_99_organizing_target_df[rel_99_cols], left_on=['UC Employee Name_std'], right_on=['Employee Lived Name_std'], suffixes=('_doc', '_99'))


In [109]:
email_unmatch_name_match_df['matched_on'] = 'UC Employee Name'

In [110]:
email_unmatch_name_match_df.shape

(216, 48)

In [111]:
email_unmatch_name_match_df['Sheet ID'].nunique()

208

In [112]:
email_unmatch_name_match_df.to_excel('email_unmatch_name_match_perfect_univ.xlsx', index=False)

In [115]:
set(email_unmatch_name_match_df.columns) - set(doct_email_matches_df.columns)

set()

In [114]:
email_unmatch_name_match_df = email_unmatch_name_match_df.drop(columns=['Employee Lived Name_std', 'UC Employee Name_std'])

In [116]:
doc_matched_names_emails_df = pd.concat([doct_email_matches_df, email_unmatch_name_match_df])
doc_matched_names_emails_df.shape

(7635, 46)

In [117]:
doc_no_name_no_email_match_df = doctor_list_df[~doctor_list_df['Sheet ID'].isin(doc_matched_names_emails_df['Sheet ID'])]
doc_no_name_no_email_match_df.shape

(1277, 34)

In [129]:
len(doc_matched_names_emails_df['Sheet ID'].unique())

7546

Match the name and email non matches against the greater 99 list based on emails


In [ ]:
doc_no_name_no_email_match_df = standardize_emails(doc_no_name_no_email_match_df, 'UC Email')
doc_no_name_no_email_match_df = standardize_emails(doc_no_name_no_email_match_df, 'UC Email 2')
latest_99_df = standardize_emails(latest_99_df, 'EE Work Email Address')

/tmp/ipykernel_1738/1727626450.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[email_col] = (
/tmp/ipykernel_1738/1727626450.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[email_col] = (


In [ ]:
doc_unmatched_in_99_email1_matches = doc_no_name_no_email_match_df.merge(latest_99_df, left_on=['UC Email'], right_on=['EE Work Email Address'], suffixes=('_doc', '_99'))

In [ ]:
doc_unmatched_in_99_email2_matches = doc_no_name_no_email_match_df.merge(latest_99_df, left_on=['UC Email 2'], right_on=['EE Work Email Address'], suffixes=('_doc', '_99'))

In [ ]:
doc_unmatched_99_email_matched_df = pd.concat([doc_unmatched_in_99_email1_matches, doc_unmatched_in_99_email2_matches])
doc_unmatched_99_email_matched_df = doc_unmatched_99_email_matched_df.drop_duplicates(subset=['Sheet ID'])
doc_unmatched_99_email_matched_df.shape

(1249, 59)

In [ ]:
doc_unmatched_99_email_matched_df = doc_unmatched_99_email_matched_df[doc_unmatched_99_email_matched_df['EE Work Email Address'] != "nan"]
doc_unmatched_99_email_matched_df.shape

(1113, 59)

In [ ]:
doc_unmatched_99_email_matched_df.columns

Index(['Campus', 'Job Title', 'Department', 'First/Preferred Name',
       'Middle Name', 'Last Name', 'Pronouns', 'UC Provided Work Location',
       'Building', 'Room', 'Location notes: remote, office hours, etc.',
       'Personal Email', 'Personal Email 2', 'Personal Email 3', 'UC Email',
       'UC Email 2', 'UC Email 3', 'Cell Phone', 'Home Phone', 'Work Phone',
       'LinkedIn URL', 'Other Website(s)', 'Home Address', 'Age',
       'Research Notes', 'Hire Date', 'Job Expected End Date_doc',
       'Employee Class Description_doc', 'FTE_doc', 'Job Hourly Rate_doc',
       'Employee Relation Description_doc', 'Location/Business Unit_doc',
       'UC Employee Name', 'Sheet ID', 'Location/Business Unit_99',
       'Employee Lived Name', 'Job Code', 'Job Code Description',
       'Most Recent Date of Hire in Empl Rcd', 'Job Expected End Date_99',
       'Employee Class', 'Employee Class Description_99', 'FTE_99',
       'Job Hourly Rate_99', 'Annual Salary', 'Department ID',
       

Add the column indicating if the doctors unmatched on name and email are on other titles in 99 list

In [ ]:
cols_to_keep = ['Sheet ID', 'Location/Business Unit_99',
       'Employee Lived Name', 'Job Code', 'Job Code Description',
       'Most Recent Date of Hire in Empl Rcd','Employee Class Description_99', 'FTE_99',
        'Department Description','EE Work Email Address','Employee Relation Description_99']

In [ ]:
doc_no_match_target_pop_99_details_df = doc_no_name_no_email_match_df.merge(doc_unmatched_99_email_matched_df[cols_to_keep], on='Sheet ID', how='left')

In [ ]:
doc_no_match_target_pop_99_details_df.to_excel('doc_no_match_target_pop_99_details.xlsx', index=False)

List the new comers

In [ ]:
new_comers = latest_99_organizing_target_df[~latest_99_organizing_target_df['index'].isin(doc_matched_names_emails_df['index'])]
new_comers.shape

(101, 27)

In [ ]:
new_comers.to_excel('new_comers.xlsx', index=False)

Find the matches on multiple sheetIDs

In [123]:
index_dupes = doc_matched_names_emails_df.groupby('index')['Sheet ID'].nunique().loc[lambda x: x > 1]

In [124]:
index_dupes

,Sheet ID
index,
29839,2
30378,2
46780,2
46883,2
46904,2
47111,2
47180,2
47684,2
47948,2


In [119]:
len(index_dupes)

27

In [ ]:
sheet_Id_match_dupes = doc_matched_names_emails_df[doc_matched_names_emails_df['index'].isin(index_dupes.index)].sort_values('Sheet ID')

In [ ]:
sheet_Id_match_dupes.to_excel('sheet_Id_match_dupes.xlsx', index=False)